# [16.9] Blind prediction: change the learning game
**How much does exact Data Shapley credit for the rare positive example change when coalition training divides by coalition size?**
Predict **Δφ7 = φ7(subset mean) − φ7(fixed n)** in mean-validation BCE nats.
A positive prediction means more credit under subset-mean training; a negative one means less.
The estimate may be any finite number. There is no imposed [-1, 1] range.

Both games start logistic regression from zero and take 40 gradient-descent steps at learning rate 0.2.
For a nonempty coalition S, the only change is dividing summed training gradients by 8 versus |S|.
Empty S leaves the model at zero. Validation loss always averages the same eight validation rows.
These are **different learning algorithms/games**, not competing estimators of one fixed game.

You will implement batched coalition training, utility, exact Shapley and coalition-size marginal curves;
then forecast, run paired conditions, see controls and compare the evidence with your explanation.
The shared full-coalition endpoint does not determine how its total utility is distributed.
No target result or figure is prefilled. Practice tests use a separate small invented fixture.

Exposure is `unknown`, `seen`, or `self_reported_not_seen`. Opening equivalent outcomes makes this practice;
a new fingerprint never proves you are unseen. This is an exploratory analytic toy with fresh tiny-model
training after commitment, not independent TEST evaluation or a mastery certificate.


In [ ]:
from pathlib import Path
import sys
root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'mastery_hub').is_dir())
if str(root) not in sys.path: sys.path.insert(0,str(root))
from mastery_hub.units.shapley_prediction import lesson
import torch as t
from IPython.display import display, Code
t.set_num_threads(1)
CASE = dict(lesson.CASE)
print(CASE)
# Existing installed environment; explicit CPU float64, no downloads or GPU calls.


## Know the players before forecasting
The printed case contains every ordered training/validation row and label. Players 0 and 1 are exact
duplicates. Player 6 has the same features but the opposite label. Player 7 is the rare positive at (0.2, 2.5).
All rows remain fixed. No outcome is needed to inspect the inputs.

A coalition is a bitmask m in 0…255: bit i selects training row i. `masks[m,i]` is its 0/1 membership.
For each coalition fit a separate two-weight plus scalar-bias logistic regressor. Vectorizing all 256
models avoids Python loops over training runs without changing the game.


## Exercise 1: train all coalition models visibly
Implement `train_coalitions(x, y, masks, steps, lr, policy)` returning `(K,d+1)` final states,
weights first, bias last. Allocate zeros with `x.new_zeros`. Logits have shape `(K,n)`.
The per-example derivative is `sigmoid(logits)-y`, multiplied by `masks`. Sum weighted errors
for bias and multiply by x for weights. Divide by n under `fixed_n`; under `subset_mean`,
divide by each mask's sum clamped to at least one. Do not mutate x/y/masks.
<details><summary>Help</summary>The empty mask gives zero summed gradients in every step.
Use one old logits/errors tensor to update both weights and bias. `errors @ x` computes all weight gradients.
The independent fixture includes both denominator policies, empty/singleton/full coalitions, zero steps and zero LR.</details>


In [ ]:
TRAINING_SOURCE = '''
def train_coalitions(x, y, masks, steps, lr, policy):
    raise NotImplementedError("Implement train_coalitions")
'''
ns = lesson.method_namespace(TRAINING_SOURCE, complete=False)
print(lesson.practice_test(ns, 'training'))


Expected: `training: independent practice fixture passed`. These tests never evaluate the eight-player target.


## Exercise 2: validation utility with a fixed denominator
Implement `coalition_utilities(states, val_x, val_y)`, returning one utility per fitted state.
Compute mean validation BCE across the same validation rows for every coalition, then subtract from log(2),
the zero model's loss. Never divide validation loss by training-coalition size.
<details><summary>Help</summary>`states[:, :-1] @ val_x.T + states[:, -1:]` gives logits.
Unreduced stable BCE, averaged on the validation axis, avoids overflow and accidental cross-model averaging.</details>


In [ ]:
UTILITY_SOURCE = '''
def coalition_utilities(states, val_x, val_y):
    raise NotImplementedError("Implement coalition_utilities")
'''
ns = lesson.method_namespace(UTILITY_SOURCE, complete=False)
print(lesson.practice_test(ns, 'utility'))


Expected: `utility: independent practice fixture passed`. These tests never evaluate the eight-player target.


## Exercise 3: exact Shapley allocation
Implement `exact_shapley(values, n)`. For every player i and predecessor mask S not containing i,
add `[v(S∪{i})−v(S)] / [n × comb(n−1, |S|)]`. Require the complete bitmask table of length 2**n.
Retain `v(empty)`; it need not be zero for an arbitrary practice game.
<details><summary>Help</summary>`mask.bit_count()` is coalition size; `mask | (1 << i)` inserts i.
The fixture averages all six orderings of a three-player interacting game independently.
An additional four-player fixture contains null players.</details>


In [ ]:
SHAPLEY_SOURCE = '''
def exact_shapley(values, n):
    raise NotImplementedError("Implement exact_shapley")
'''
ns = lesson.method_namespace(SHAPLEY_SOURCE, complete=False)
print(lesson.practice_test(ns, 'shapley'))


Expected: `shapley: independent practice fixture passed`. These tests never evaluate the eight-player target.


## Exercise 4: mean marginal by predecessor size
Implement `marginal_by_size(values, n, player)`, an n-vector. At each k, average that player's marginal
contribution over all `comb(n−1,k)` predecessor coalitions of size k. The unweighted mean of these n strata
must equal that player's exact Shapley value. This diagnoses where an algorithm change affects the allocation.
<details><summary>Help</summary>Accumulate each predecessor marginal into index `mask.bit_count()`
after dividing by the number of predecessors at that size. Do not weight sizes by their number of coalitions again.</details>


In [ ]:
STRATA_SOURCE = '''
def marginal_by_size(values, n, player):
    raise NotImplementedError("Implement marginal_by_size")
'''
ns = lesson.method_namespace(STRATA_SOURCE, complete=False)
print(lesson.practice_test(ns, 'strata'))


Expected: `strata: independent practice fixture passed`. These tests never evaluate the eight-player target.


## Exercise 5: assemble the result from cached values
Implement `assemble_result(states, values, n, player)`. Return JSON-native fields `states`, `utilities`,
`shapley`, `marginals7`, and `loo`. Use your exact/strata functions; `marginals7` names the selected player's
curve (player 7 in the target; player 2 in the practice fixture). LOO_i is `v(full)−v(full without i)`.
LOO is a baseline from the same cached table and need not allocate total utility correctly.
<details><summary>Help</summary>Use `.tolist()` for tensors. `full ^ (1 << i)` removes i from a full mask.
Every model state and coalition utility must remain in ascending bitmask order for reproducible evidence.</details>


In [ ]:
ASSEMBLY_SOURCE = '''
def assemble_result(states, values, n, player):
    raise NotImplementedError("Implement assemble_result")
'''
ns = lesson.method_namespace(ASSEMBLY_SOURCE + "\n\n" + SHAPLEY_SOURCE + "\n\n" + STRATA_SOURCE, complete=False)
print(lesson.practice_test(ns, 'assembly'))


Expected: `assembly: independent practice fixture passed`. These tests never evaluate the eight-player target.


## Exposed DEV ladder — a denominator changes a step (8–12 minutes)
Use the separate three-row training fixture below, one gradient step at lr=.2 from zero.
For singleton {0}, sigmoid(0)−y0=.5 and x0=[.4,−1]. Hand-predict the two weights and bias
under denominator 3 versus denominator 1, then predict their state-norm ratio.

Choose **A:** changing the denominator scales every coalition identically, including full;
or **B:** smaller coalitions receive a larger step, while the full coalition is unchanged.
What observation rules one explanation out? Plot all three players' allocations as well:
a scaled parameter step need not imply equally scaled validation utility or Shapley credit.
These are public three-player DEV observations, not the eight-player target.
Confidence below is an **unscored self-rating**, not a probability of an exact numeric value.


In [ ]:
dev_forecast_ratio = 3.  # Hand prediction: singleton state norm, subset_mean / fixed_n
dev_self_rating = .7  # unscored self-rating
dev_explanation = 'B'
dev_disconfirming_observation = 'Full models agree while singleton states differ.'


In [ ]:
import matplotlib.pyplot as plt
dev_source = '\n\n'.join([TRAINING_SOURCE,UTILITY_SOURCE,SHAPLEY_SOURCE,STRATA_SOURCE,ASSEMBLY_SOURCE])
dev_ns = lesson.method_namespace(dev_source)
dev_x = t.tensor([[.4,-1.],[1.2,.3],[-.5,.8]],dtype=t.float64)
dev_y = t.tensor([0.,1.,1.],dtype=t.float64)
dev_masks = t.tensor([[(m>>i)&1 for i in range(3)] for m in range(8)],dtype=t.float64)
dev_results = {}
for policy in ('fixed_n','subset_mean'):
    states = dev_ns['train_coalitions'](dev_x,dev_y,dev_masks,1,.2,policy)
    values = dev_ns['coalition_utilities'](states,dev_x,dev_y)
    credits = dev_ns['exact_shapley'](values,3)
    dev_results[policy] = (states,values,credits)
dev_zero_states = dev_ns['train_coalitions'](dev_x,dev_y,dev_masks,1,0.,'subset_mean')
dev_zero_credit = dev_ns['exact_shapley'](dev_ns['coalition_utilities'](dev_zero_states,dev_x,dev_y),3)
fig,axes = plt.subplots(1,2,figsize=(11,3),layout='constrained')
for policy,(states,values,credits) in dev_results.items():
    axes[0].plot(range(8),states.norm(dim=1).tolist(),'o-',label=policy)
    axes[1].plot(range(3),credits.tolist(),'o-',label=policy)
axes[0].plot(range(8),dev_zero_states.norm(dim=1).tolist(),'--',label='Zero learning rate')
axes[1].plot(range(3),dev_zero_credit.tolist(),'--',label='Zero learning rate')
axes[0].set(xlabel='Coalition bitmask',ylabel='State norm',title='DEV one-step learning',xticks=range(8))
axes[1].set(xlabel='DEV player',ylabel='Validation BCE improvement (nats)',title='Exact DEV allocation',xticks=range(3))
for ax in axes: ax.legend()
display(fig)
dev_observed_ratio = float(dev_results['subset_mean'][0][1].norm()/dev_results['fixed_n'][0][1].norm())
print('DEV singleton norm ratio:',dev_observed_ratio)
print('DEV ratio absolute forecast error:',abs(dev_observed_ratio-dev_forecast_ratio))
t.testing.assert_close(dev_results['subset_mean'][0][7],dev_results['fixed_n'][0][7])
assert dev_zero_states.count_nonzero() == 0


### Worked DEV reflection / self-check
The singleton states are [−.013333,.033333,−.033333] and [−.04,.1,−.1]; their norm ratio
is 3. A mistaken forecast of 1 has absolute ratio error 2. The full-coalition state is identical:
that observation rules A out. A good reflection cites singleton AND full controls, then explains
why nonlinear BCE and allocations over all predecessor sizes prevent multiplying every credit by 3.
The zero-LR allocation stays zero. These checks do not establish that either learning game is better.

### Try It Yourself — changed-condition DEV transfer
Change ONLY the number of training steps from one to two. Freeze data, masks, zero
initialization, lr=.2, validation denominator and policies. Predict the singleton state-norm
ratio before running. Does the one-step explanation imply exactly3 after a second nonlinear
step? Distinguish local gradient scaling from final model scaling.
The next cell trains a new DEV condition; it does not just inspect an already stored player.


In [ ]:
dev_transfer_steps = 2
dev_transfer_forecast = 3.  # Deliberate one-step extrapolation; replace before running.
dev_transfer_results = {}
for policy in ('fixed_n','subset_mean'):
    states=dev_ns['train_coalitions'](dev_x,dev_y,dev_masks,dev_transfer_steps,.2,policy)
    values=dev_ns['coalition_utilities'](states,dev_x,dev_y)
    dev_transfer_results[policy]=(states,values,dev_ns['exact_shapley'](values,3))
dev_transfer_observed=float(dev_transfer_results['subset_mean'][0][1].norm()/dev_transfer_results['fixed_n'][0][1].norm())
print('DEV two-step singleton ratio:',dev_transfer_observed,'absolute forecast error:',abs(dev_transfer_observed-dev_transfer_forecast))
assert abs(dev_transfer_observed-3.) > 1e-8
t.testing.assert_close(dev_transfer_results['subset_mean'][0][7],dev_transfer_results['fixed_n'][0][7])
fig,axes=plt.subplots(1,2,figsize=(11,3),layout='constrained')
axes[0].plot([1,2],[dev_observed_ratio,dev_transfer_observed],'o-',label='Observed singleton norm ratio')
axes[0].axhline(3,color='gray',linestyle='--',label='Exact step-1 extrapolation')
axes[0].set(xlabel='Training steps',ylabel='subset_mean / fixed_n state norm',title='Observed DEV duration transfer',xticks=[1,2])
for policy in ('fixed_n','subset_mean'):
    axes[1].plot(range(3),dev_transfer_results[policy][2].tolist(),'o-',label=policy+' two steps')
axes[1].plot(range(3),dev_zero_credit.tolist(),'--',label='Zero-LR control')
axes[1].set(xlabel='DEV player',ylabel='Shapley credit (BCE nats)',title='New DEV allocations',xticks=range(3))
for ax in axes:ax.legend()
display(fig)
dev_revision = ''  # Why does a second gradient see different states under the two policies?


<details><summary>DEV transfer self-check</summary>
The ratio is no longer exactly3. At step2 the policies evaluate sigmoid at different
parameters: n/|S| scales a gradient at the same state, not every final state after nonlinear
training. Cite the measured discrepancy, the matching full-coalition control, and whether
allocation changed. No universal claim about rare-point credit follows.</details>


## Freeze exact inputs, then forecast
Finish each implementation and its separate fixture first. Freezing hashes the exact ordered inputs,
initial state, coalition masks, case rules, method text, helper/test sources and environment; it performs **no training or utility evaluation**.
Record your own numerical estimate, confidence and causal reason. Think about how a player's marginal
changes in small coalitions versus large ones. Name an observation that would challenge your explanation.
Editing source/runtime after freezing requires a new commitment. Local hashes detect accidental drift;
they are not exam security or proof of exposure history.


In [ ]:
METHOD_SOURCE = '\n\n'.join([TRAINING_SOURCE, UTILITY_SOURCE, SHAPLEY_SOURCE, STRATA_SOURCE, ASSEMBLY_SOURCE])
identity = lesson.freeze_inputs(METHOD_SOURCE)
print('Frozen exact inputs and source; no target model has been trained.')


In [ ]:
prediction = None  # Your finite Δφ7 in mean-validation BCE nats
confidence = None  # Unscored reasoning self-rating in [0,1]; not a probability
reasoning = ''  # Causal explanation and what could contradict it
prior_exposure = 'unknown'  # unknown / seen / self_reported_not_seen
commitment = lesson.commit_prediction(identity, METHOD_SOURCE, prediction, confidence, reasoning, prior_exposure)
print('Committed. No target training or utility evaluation has run.')


## Run the fixed paired experiment after commitment
The explicit run repeats the independent practice checks, then fits four 256-coalition games:
both denominator policies and their label-0↔6 relocation controls. Two additional zero-learning-rate
batches check inert training. Full-coalition states/utility must match, empty models stay zero,
duplicates share credit, exact allocations satisfy efficiency and relocation swaps only the corresponding credits.
There is one cooperative 120-second CPU budget, checked between stages; slow learner functions are
rejected on return rather than forcibly interrupted. No GPU/download is used. Preserve exceptions as failed attempts.
Reference implementation text is optional after commitment and contains no measured outcome.
Adopting changed code requires freezing again and creating a separate forecast.


In [ ]:
# Optional postcommit implementation help, no training:
# display(Code(lesson.reveal_methods(commitment), language='python'))


In [ ]:
try:
    receipt = lesson.run_experiment(commitment, METHOD_SOURCE)
except lesson.ExperimentFailure as failed:
    failed_record = failed.record_text  # Preserves actual measurements and original commitment.
    print(lesson.replay_failure(failed_record))
    # Optional: lesson.export_record(failed_record, Path.home()/'shapley-failed-attempt-1.json')
    raise  # Stop here; replay/export failure without rerunning training.
observation = lesson.replay(receipt)
display(lesson.plot_receipt(receipt))
print('Observed Δφ7:', observation['delta_phi7'], 'nats')
print('Absolute forecast error:', observation['absolute_forecast_error'])
print('Controls:', observation['controls'])


## Explain the allocation, not just the winner
Read the paired bars alongside the marginal-by-size curves. Which predecessor sizes changed most?
Does that support your explanation? Both games share a full-coalition model/utility, but their allocation
across players can differ. Compare LOO without treating it as an efficiency-preserving allocation.
Check the empty/zero-LR, duplicate, relocation and efficiency controls before interpreting the forecast error.
A passing control does not prove the game is a good definition of real-world data value.

This is an eight-point unregularized logistic toy, 40 fixed steps, one learning rate and one validation set.
No random cohort, confidence interval, independent TEST performance, universal pruning conclusion or human
mastery claim follows from this exact calculation. Different training schedules define further games.
The [canonical 16.9 reference](../../../chapter16_shapley_attribution_baselines/exercises/part9_data_shapley/16.9_Data_Shapley_in_One_Training_Run_solutions.ipynb)
offers retraining and in-run accounting as distinct concepts; this challenge isolates the retraining game's denominator only.
After recording your forecast, read Ghorbani & Zou's [Data Shapley: Equitable Valuation of Data for Machine Learning](https://arxiv.org/abs/1904.02868).
Its retraining-based data valuation is distinct from the In-Run attribution later in the canonical section.


In [ ]:
# Try It Yourself: inspect another stored player without fitting anything.
player = 0
obs = lesson.replay(receipt)
for policy in ('fixed_n','subset_mean'):
    result = obs['results'][policy]
    print(policy, 'Shapley:', result['shapley'][player], 'LOO:', result['loo'][player])
reflection = ''  # What matched, what failed, and which control informed the explanation?
transfer_question = ''  # Change ONE condition and retain metric/units/denominator definitions.
next_prediction = ''  # New forecast BEFORE a separately authorized experiment.
transfer_confidence = None
transfer_reasoning = ''
# attempt = lesson.reflect(receipt, reflection, transfer_question, next_prediction, transfer_confidence, transfer_reasoning)


## Keep forecast, result and transfer together
Reflection wraps the exact original receipt, preserving the original forecast and machine observation.
Save only by explicit opt-in. Export creates a new private file (0600) and refuses to overwrite.
Replay/plotting read stored values; they evaluate no student code, training loop, torch model or current environment.
Seals validate structure and accidental drift, not an adversarially edited local record.


In [ ]:
# Optional exports to explicit new filenames:
# lesson.export_record(receipt, Path.home()/'shapley-prediction-receipt-1.json')
# lesson.export_attempt(attempt, Path.home()/'shapley-prediction-reflection-1.json')
# Later, model-free replay:
# saved = Path.home().joinpath('shapley-prediction-receipt-1.json').read_text()
# display(lesson.plot_receipt(saved))
